# Home Credit — Pipeline Notebook (Tutor Session)

**Plan:** [`arson/docs/TUTOR_PLAN.md`](../docs/TUTOR_PLAN.md)

**Goal:**
```
TRAIN (307511 × 122)   TEST (48744 × 121)
         └────── SAME PIPELINE (fit on train only) ──────┘
                        │
              X_train_processed   X_test_processed
                 (n × k)             (n × k)   ← same k
```

**Rules (Socratic hybrid):**
1. I (tutor) set a task.
2. **You write the code / answer first.**
3. I correct, then show reference only if needed.


## Step 1 — Data Audit

### Task 1.1 (setup, filled for you)
Load both CSVs and print their shapes.


In [16]:
import pandas as pd
import numpy as np
from arson.configs.settings import project_dir

p_dir = project_dir()

RAW = p_dir.RAW_DIR

df_train = pd.read_csv(f"{RAW}/application_train.csv")
df_test = pd.read_csv(f"{RAW}/application_test.csv")

print("train:", df_train.shape)
print("test :", df_test.shape)


train: (307511, 122)
test : (48744, 121)


### ❓ Q1 — Column parity (you answer in the cell below)

1. Train has 122 columns, test has 121. **Which single column differs, and why does it exist only in train?**
2. Write code that proves the remaining 121 columns are **identical in name and order**.

*Hint: set difference both ways, plus an order check.*


In [17]:
# Your attempt:
# Train has 121 + 1 target var, however test data does not have that target col, it only exist in train data, so we can train and test on training data
# test on test set. 

for col in df_train.columns:
    if col not in df_test.columns:
        print(col)

TARGET


### ❓ Q2 — Target balance (maps to test.ipynb Q1, but answer fresh)

1. Compute `TARGET` value counts and normalized percentages.
2. A lazy model predicts `0` for everyone. Write the one line that computes its accuracy.
3. What would its **recall for class 1** be? Write the code (or 0 if trivial).


In [18]:
# Your attempt:
df_train['TARGET'].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

### ❓ Q3 — Missingness triage

`arson.txt` has per-column missing counts for train. Load it (or compute yourself) and answer:

1. How many columns have **0**, **>30%**, **>50%** missing?
2. Pick one >50% column (e.g. `OWN_CAR_AGE`). Should we drop it, impute it, or keep it with a missing-indicator? **Commit to a choice and justify in a comment** — we'll test it later.

*You may peek at mentor's EDA but do the counting yourself.*


In [19]:
# Your attempt:
missing = df_train.isnull().sum().to_frame('missing_value_count')
missing['percentage'] = (df_train.isnull().sum()/len(df_train) )* 100

missing.sort_values(by='percentage',ascending=False)

print(missing[missing['percentage'] > 50].shape)
print(missing[missing['percentage']> 30].shape)

missing[missing['percentage'] > 50].sample()



(41, 2)
(50, 2)


,missing_value_count,percentage
ENTRANCES_AVG,154828,50.348768


In [20]:
df_columns_desc_dataset = p_dir.RAW_DIR/'col_description.csv'

df_col_desc = pd.read_csv(df_columns_desc_dataset,encoding='latin1',index_col=0)
df_col = df_col_desc[df_col_desc['Table']=='application_{train|test}.csv']
df_col[df_col['Row']=='APARTMENTS_MEDI']


,Table,Row,Description,Special
75,application_{train|test}.csv,APARTMENTS_MEDI,Normalized information about building where th...,normalized


In [21]:
df_train['APARTMENTS_MEDI'].value_counts().head(10)#

APARTMENTS_MEDI
0.0833    7109
0.0625    6687
0.0937    4622
0.0729    4211
0.0083    3562
0.0167    3098
0.1041    3097
0.1499    2992
0.0125    2801
0.0749    2559
Name: count, dtype: int64

In [22]:
## lets see is there any relation with our target col
apartments_medI_with_target = df_train.groupby('APARTMENTS_MEDI')['TARGET'].mean().to_frame()
apartments_medI_with_target[apartments_medI_with_target['TARGET']>0.3]

# this col is so random, i dnt think that this col has any connection with our target variable. 
# bout the missing value 

,TARGET
APARTMENTS_MEDI,
0.1785,0.333333
0.2607,0.333333
0.3055,0.400000
0.3263,0.500000
0.3367,0.333333
0.5116,0.333333
0.6058,0.333333
0.6766,0.333333
0.7203,0.333333


In [23]:
df_train[df_train['APARTMENTS_MEDI'].isna()].sample(4)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
277841,421896,0,Cash loans,F,N,Y,0,103500.0,225000.0,8613.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
240420,378450,1,Cash loans,F,Y,Y,2,180000.0,859212.0,30991.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
147325,270813,0,Cash loans,F,Y,N,0,67500.0,107820.0,7798.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
294382,441050,0,Cash loans,F,N,Y,2,72000.0,75636.0,3807.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
cat_col = df_train.select_dtypes('str')
num_col = df_train.select_dtypes(['int64','float64'])
print(cat_col.shape)
print(num_col.shape)

(307511, 16)
(307511, 106)


In [25]:
col_type=df_train.dtypes.to_frame()
col_type[0].value_counts()
col_type[col_type[0] == 'str']

,0
NAME_CONTRACT_TYPE,str
CODE_GENDER,str
FLAG_OWN_CAR,str
FLAG_OWN_REALTY,str
NAME_TYPE_SUITE,str
NAME_INCOME_TYPE,str
NAME_EDUCATION_TYPE,str
NAME_FAMILY_STATUS,str
NAME_HOUSING_TYPE,str
OCCUPATION_TYPE,str


In [26]:
cat_col
cat_col_names = df_train.select_dtypes('str').columns
num_col_names = df_train.select_dtypes('int64','float64').columns

In [27]:
num_col_names

Index(['SK_ID_CURR', 'TARGET', 'CNT_CHILDREN', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
       'DAYS_ID_PUBLISH', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
       'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'REGION_RATING_CLIENT',
       'REGION_RATING_CLIENT_W_CITY', 'HOUR_APPR_PROCESS_START',
       'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION',
       'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'FLAG_DOCUMENT_2',
       'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5',
       'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8',
       'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11',
       'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14',
       'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17',
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21'],
      dtype='str')

In [28]:
df_col[df_col['Row'].str.startswith('FLAG')]

,Table,Row,Description,Special
7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
8,application_{train|test}.csv,FLAG_OWN_REALTY,Flag if client owns a house or flat,NaN
25,application_{train|test}.csv,FLAG_MOBIL,"Did client provide mobile phone (1=YES, 0=NO)",NaN
26,application_{train|test}.csv,FLAG_EMP_PHONE,"Did client provide work phone (1=YES, 0=NO)",NaN
27,application_{train|test}.csv,FLAG_WORK_PHONE,"Did client provide home phone (1=YES, 0=NO)",NaN
28,application_{train|test}.csv,FLAG_CONT_MOBILE,"Was mobile phone reachable (1=YES, 0=NO)",NaN
29,application_{train|test}.csv,FLAG_PHONE,"Did client provide home phone (1=YES, 0=NO)",NaN
30,application_{train|test}.csv,FLAG_EMAIL,"Did client provide email (1=YES, 0=NO)",NaN
99,application_{train|test}.csv,FLAG_DOCUMENT_2,Did client provide document 2,NaN
100,application_{train|test}.csv,FLAG_DOCUMENT_3,Did client provide document 3,NaN


In [42]:
flag_doc = df_train.columns.str.startswith('FLAG_DOCUMENT')
flag_doc_col = df_train.columns[flag_doc]
# df_train[flag_doc_col]
df_train.shape
df_train.drop(flag_doc_col,axis=1,inplace=True)
df_train.shape

df_test.drop(flag_doc_col,axis=1,inplace=True)

In [43]:
# re defining the cat and num col
cat_col = df_train.select_dtypes('str')
num_col = df_train.select_dtypes(['int64','float64'])

In [44]:
num_col.shape

(307511, 86)

In [30]:
df_train.describe()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,307511.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.573909,5.383962e+05,0.020868,-16036.995067,63815.045904,...,0.008130,0.000595,0.000507,0.000335,0.006402,0.007000,0.034362,0.267395,0.265474,1.899974
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737315,3.694465e+05,0.013831,4363.988632,141275.766519,...,0.089798,0.024387,0.022518,0.018299,0.083849,0.110757,0.204685,0.916002,0.794056,1.869295
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-2760.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1213.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-289.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,365243.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


In [31]:
apartments_medI_with_target

,TARGET
APARTMENTS_MEDI,
0.0000,0.080415
0.0010,0.050251
0.0016,0.166667
0.0021,0.082921
0.0026,0.000000
...,...
0.9899,0.000000
0.9910,0.000000
0.9972,0.000000


In [32]:
df_columns_desc_dataset

PosixPath('/home/arson/birdy/hands-on/arson/home-risk-kaggle/data/raw/col_description.csv')

In [33]:

df_train['YEARS_BUILD_AVG'].value_counts().to_frame().sort_values(by='count',ascending=False)

,count
YEARS_BUILD_AVG,
0.8232,2999
0.8164,2864
0.8028,2848
0.7280,2802
0.7348,2761
...,...
0.0616,2
0.0548,2
0.1976,2


In [55]:
df_train.groupby('YEARS_BUILD_AVG')['TARGET'].mean().sort_index(ascending=False)

YEARS_BUILD_AVG
1.0000    0.063584
0.9932    0.083682
0.9864    0.055976
0.9796    0.049618
0.9728    0.044280
            ...   
0.0208    0.000000
0.0140    0.000000
0.0072    0.250000
0.0004    0.000000
0.0000    0.068627
Name: TARGET, Length: 149, dtype: float64

In [45]:
df_col

,Table,Row,Description,Special
1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
...,...,...,...,...
120,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_DAY,Number of enquiries to Credit Bureau about the...,NaN
121,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_WEEK,Number of enquiries to Credit Bureau about the...,NaN
122,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_MON,Number of enquiries to Credit Bureau about the...,NaN
123,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_QRT,Number of enquiries to Credit Bureau about the...,NaN


In [ ]:
df_col[df_col['Row'] == 'YEARS_BUILD_AVG']

In [57]:
df_train['REGION_RATING_CLIENT'].value_counts()

REGION_RATING_CLIENT
2    226984
3     48330
1     32197
Name: count, dtype: int64

In [62]:
df_train.groupby('REGION_RATING_CLIENT')['TARGET'].agg({
    'count','mean'
})

,mean,count
REGION_RATING_CLIENT,,
1,0.048203,32197
2,0.078891,226984
3,0.111028,48330


In [59]:
df_train['REGION_POPULATION_RELATIVE'].value_counts()

REGION_POPULATION_RELATIVE
0.035792    16408
0.046220    13442
0.030755    12163
0.025164    11950
0.026392    11601
            ...  
0.001417      467
0.001333      235
0.000533       39
0.000938       28
0.000290        2
Name: count, Length: 81, dtype: int64

In [61]:
df_train.groupby('CODE_GENDER')['TARGET'].agg(
    {
        'count','mean'
    }
)

,mean,count
CODE_GENDER,,
F,0.069993,202448
M,0.101419,105059
XNA,0.000000,4


In [73]:
df_train['FLAG_OWN_REALTY'] = df_train['FLAG_OWN_REALTY'].map({
    'Y' : 1,
    'N': 0
})
df_test['FLAG_OWN_REALTY'] = df_test['FLAG_OWN_REALTY'].map({
    'Y': 1,
    'N': 0
})
# df_train['FLAG_OWN_REALTY']

In [72]:
df_train.groupby('FLAG_OWN_REALTY')['TARGET'].agg({
    'count','mean'
})

,mean,count
FLAG_OWN_REALTY,,


In [ ]:
df_train.groupby('F')

### ❓ Q4 — Why the mentor's approach broke (maps to test Q4 — leakage & reuse)

Mentor did: `df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]` directly on the whole `df`, every time.

Answer in plain words (markdown or comments):
1. What goes wrong when you try to apply those same manual lines to `df_test` — why can the result end up a **different shape**?
2. What must `fit` learn from train only (medians, categories, etc.)?
3. Correct order: `fit` on _____, `transform` on _____ and _____.



In [34]:
# Your attempt (or write answers as comments):
# 1. i dont think that shape would be different, or we will get different result for test set. 
# 2. to prevent data leakage, if we use fit on our test set, now model knows the the mean,median of our test data
# 3. fit on train, transform on test and idk.

### Setup reminder (run after your attempts)
Make sure `TARGET` distributions and column-parity results look sane before moving on.


In [35]:
# Sanity reference — run only after you've attempted Q1-Q3 yourself
assert set(df_train.columns) - set(df_test.columns) == {"TARGET"}
assert set(df_test.columns) - set(df_train.columns) == set()
assert [c for c in df_train.columns if c != "TARGET"] == list(df_test.columns)

print("TARGET train balance:")
print(df_train["TARGET"].value_counts(normalize=True).round(4))
print("\nBaseline accuracy (always predict 0):",
      round((df_train["TARGET"] == 0).mean(), 4))


TARGET train balance:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64

Baseline accuracy (always predict 0): 0.9193
